In [0]:
from pyspark.sql.functions import (
    col, avg, count, min, max, round, stddev, expr
)
from pyspark.sql.window import Window


silver_df = spark.read.format("delta").table("final_project.Silver_table")

global_filtered_df = silver_df.filter(
    (col("Price") >= 10000) & (col("Price") <= 10000000) &
    (col("Year") >= 1990) & (col("Year") <= 2026)
)


window_spec = Window.partitionBy("Brand", "Model")

stats_df = (
    global_filtered_df
    .withColumn("avg_model_price", avg("Price").over(window_spec))
    .withColumn("stddev_model_price", stddev("Price").over(window_spec))
    .withColumn("model_count", count("Price").over(window_spec))
)

cleaned_silver_df = stats_df.filter(
    (col("model_count") < 5) | 
    (col("stddev_model_price").isNull()) |
    (col("Price") <= (col("avg_model_price") + (3 * col("stddev_model_price"))))
).drop("avg_model_price", "stddev_model_price", "model_count")


gold_price_by_model = (
    cleaned_silver_df.groupBy("Brand", "Model")
    .agg(
        count("*").alias("listing_count"),
        round(avg("Price"), 0).alias("avg_price"),
        round(expr("percentile_approx(Price, 0.5)"), 0).alias("median_price"),
        min("Price").alias("min_price"),
        max("Price").alias("max_price"),
        round(stddev("Price"), 0).alias("stddev_price"),
        round(avg("Mileage"), 0).alias("avg_mileage")
    )
    .orderBy(col("listing_count").desc())
)

gold_price_by_model_year = (
    cleaned_silver_df.groupBy("Brand", "Model", "Year")
    .agg(
        count("*").alias("listing_count"),
        round(avg("Price"), 0).alias("avg_price"),
        round(expr("percentile_approx(Price, 0.5)"), 0).alias("median_price"),
        min("Price").alias("min_price"),
        max("Price").alias("max_price"),
        round(stddev("Price"), 0).alias("stddev_price"),
        round(avg("Mileage"), 0).alias("avg_mileage")
    )
    .orderBy(col("Brand"), col("Model"), col("Year").desc())
)
gold_market_by_location = (
    cleaned_silver_df.groupBy("Location")
    .agg(
        count("*").alias("listing_count"),
        round(avg("Price"), 0).alias("avg_price"),
        round(expr("percentile_approx(Price, 0.5)"), 0).alias("median_price")
    )
    .orderBy(col("listing_count").desc())
)

gold_price_by_year = (
    cleaned_silver_df.groupBy("Year")
    .agg(
        count("*").alias("listing_count"),
        round(avg("Price"), 0).alias("avg_price"),
        round(expr("percentile_approx(Price, 0.5)"), 0).alias("median_price"),
        round(avg("Mileage"), 0).alias("avg_mileage")
    )
    .orderBy(col("Year").desc())
)

gold_fuel_transmission = (
    cleaned_silver_df.groupBy("Fuel_type", "Transmission")
    .agg(
        count("*").alias("listing_count"),
        round(avg("Price"), 0).alias("avg_price"),
        round(expr("percentile_approx(Price, 0.5)"), 0).alias("median_price")
    )
    .orderBy(col("listing_count").desc())
)

gold_price_by_model.display()
gold_price_by_model_year.display()
gold_market_by_location.display()
gold_price_by_year.display()
gold_fuel_transmission.display()


In [0]:
gold_price_by_model.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("final_project.Gold_price_by_model")
gold_market_by_location.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("final_project.Gold_market_by_location")
gold_price_by_year.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("final_project.Gold_price_by_year")
gold_fuel_transmission.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("final_project.Gold_fuel_transmission")
